# Compile-time vs Run-time

1. 
- Compile-time runs during build: checks types, resolves templates, $\dots$, then generates machine code.
- Runtime runs whenever the program executes: performs actual logic, stack operations, and heap allocations.

2. Since run-time can ran multiple times, lots of techniques below transfer run-time to compile-time to trade more (stack register) space for better amortized total (compile + run) time.

# Input - Output

Dealing with input streams: File, String, etc

### String

##### Split String

```cpp
std::vector<std::string> split(const std::string& s, char delim) {
    std::vector<std::string> answers;

    std::stringstream ss(s);

    // keep '' tokens if e.g 'a,,b' -> ['a', '', 'b']
    std::string token;
    while(std::getline(ss, token, delim)) {
        answers.emplace_back(token);
    }

    return answers;
}
```

##### Trim String

```cpp
bool is_space(char c) { return c == ' ' || c == '\t'; }

void trim(std::string& s) {

    // left trim
    std::size_t start = 0;
    while (start < s.size() && is_space(s[start])) ++start;

    // right trim
    std::size_t end = s.size();
    while (end > start && is_space(s[end - 1])) --end;

    s = s.substr(start, end - start);
}
```

### File

##### Read file, line by line

```cpp
void parse_csv(const std::string& filename, std::vector<Course>& courses) {
    std::ifstream in_stream(filename);
    if (!in_stream) return;
    
    std::string line;
    std::getline(in_stream, line);            // read header

    while (std::getline(in_stream, line)) {  // read the rest
        auto elements = // write your 'split' function
        // Do something
    }

    return;    
}
```

##### Write file, line by line

```cpp
void write_courses(std::vector<Course> courses) {
    std::ofstream out_stream(PATH_STR);
    if (!out_stream) return;

    // Write header
    out_stream << "Title,Number of Units,Quarter" << '\n';

    // Write the rest
    for (const auto& course : courses) {
        out_stream << course << '\n';
    }

    return;
}
```

# References - Pointers

1. (ordinary) Variable: has storage and an address
2. Pointer: variable that stores an address
3. Reference:

    1. Loyal / bounded to a variable:

        a reference to x never refers something else:
        ```cpp
        int& p = x;
        p = y;      // x now stores y's value
        ```

        unless explicity declare new reference

        ```cpp
        int& p = y; // NOT OK
        int& q = y; // OK
        ```

    2. Not an address-storing object

### Reference vs Pointer

POINTER

```txt
    address 1
┌──────────────┐
│      2       │   p
└──────────────┘
       |
       v
    address 2
┌──────────────┐
│     abc      │   x
└──────────────┘
```

REFERENCE

```txt
    address 2
┌──────────────┐
│     abc      │   x
└──────────────┘
       ^
       |
       r   (another name for x)
```

##### Reference **storing a Pointer**

```txt
    address 1
┌──────────────┐
│      2       │   p
└──────────────┘
       ^
       |
       r   (another name for p)

    address 2
┌──────────────┐
│     abc      │   x
└──────────────┘
```

### Pointer specifics

##### `this` , `->` or `.`

1. `->` vs `.`
```cpp
    Node x;
    Node* p = &x;

    x.next  // x is a Node type
    p->next // same as (*p).next
```

2. `this` is a **pointer** (storing address) of the current instance of a class
```cpp
    class X {
        int value;
        public:
            void set(int v) {
                this->value = v;
            }
    };
```


##### `unique_ptr` vs `shared_ptr`

| Aspect          | `unique_ptr` (one owner)             | `shared_ptr` (many owners)          |
|:-----------------:|:--------------------------------------:|:-------------------------------------:|
| Who owns it     | Only one pointer                     | Many pointers share the same thing  |
| Copying         | You can move it; CANNOT copy it  | You can move it AND copy it                     |
| Cleanup time    | When that one owner goes away        | When the last shared owner is gone  |
| Extra cost      | Almost none                          | Extra cost to count owners          |
| Good for        | One clear lifetime                   | Sharing across places               |
| Example         | Tree or list child pointers          | Shared cache / listeners / async task   |

##### `std::string_view` vs `std::string*`

| Aspect | `std::string_view` | `std::string*` |
| :---: |:------: | :----------------: |
| Meaning | Just a pointer and a size (or two pointers) | Implies string ends<br> in null char `'\0'` |
| Null Pointer | Can't have `nullptr` | Has `nullptr` |
| Mutability | Read-only; cannot modify underlying text | Can modify |
| API | Read-only inspection/slicing:<br>`size/empty`, `operator[]`, `front/back`, `data`, `find/rfind`, `find_first/last_of`, `substr`, `starts_with/ends_with`, `remove_prefix/suffix`; no mutating | Full `std::string` API<br>(`length`, `capacity`, `push_back`, `append`, `insert`, `erase`, `resize`, etc.) |
| Copy cost | Tiny, cheap to copy | Tiny, cheap to copy |

### Smart Pointer, Unique Pointer, Weak Pointer

##### Motivation: ensuring RAII (Resource Acquisition Is Initialization) aka !memory_leak

This cause memory leak!

```cpp
void f() {
    Node* n = new Node;
    
    /*  do something that cause EXCEPTION 
        -> which is like early return)
        -> n never get `delete`d
    */

    delete n;
}
```

##### Solutions: pointer's wrapper! i.e Smart Pointers

```cpp
1. std::unique_ptr<T> p(new T);
2. std::shared_ptr<T> p(new T);
3. std::weak_ptr<T> p(new T);
```

|| Unique Pointer | Shared Pointer | Weak Pointer |
|:---:|:---:|:---:|:---:|
|What is it | The owned pointer can't be shared / copied | Multiple object instance shared (point to)<br>the same owned pointer | Similar to `shared_ptr`, <br> except it doesn't track `count`,<br>to trade "deal with circular references"|
|Example|std::unique_ptr<int> p(new int); |std::shared_ptr<int> p1(new int);<br>std::shared_ptr<int> p2 = p1;|

# Const

- There's no such thing as `const` Class

- `const` object instant:
    1. Can only call `const` member functions.
    2. All **public** member variables are treated like `const`, <br> even tho it might not be.
    3. Cannot moidfy immutable member variables 

- `const` member function:
    1. Cannot modify **immutable or non-static** data members of this object.
    2. Cannot call non-const member functions on this same object.
    3. Can modify data through pointers/references it holds <br>(unless those pointees are also `const`), <br>so `const` is shallow unless you enforce deeper constness.

### How to read
Use this rule:
1. Find the variable name.

2. Read right first, then left. 
    **Reference** `&` can't be `const` (skip right to read left if `&`)

### Example (value and pointer)

1. 
    ```cpp
    const int x;
    int const x;
    ```
    Both mean the same thing: `x` is a constant `int`.

2. 
    ```cpp
    int * const p;
    ```
    `p` is a constant pointer to an `int`


3. 
    ```cpp
    const int* p;
    int const* p;
    ```
    `p` is a pointer to a constant `int`.


3. 
    ```cpp
    const int* const p; // OK
    const int* p const; // NOT OK
    ```
    `p` is a constant pointer to a constant `int`.


4. 
    ```cpp
    const int* const& r;
    ```
    `r` is a reference to a `const` pointer to a `const int`

### Medium Example (Iterator)

1. A constant iterator to non-constant values:
    ```cpp
    const T::iterator iter = v.begin();
    *iter = 5;  // OK
    ++iter;     // ERROR
    ```

2. An iterator to contant values:
    ```cpp
    T::const_iterator iter = v.begin();
    *iter = 5   // ERROR
    ++iter;     // OK
    ```

3. Both constant iterator to constant values:

    ```cpp
    const T::const_iterator iter = v.begin();
    *iter = 5;  // ERROR
    ++iter;     // ERROR
    ```

### Harder Example

![const-howto-read](../_images/const-how-to-read.png)

- `& param` -> `param` is a reference to ...
- `*` -> the thing being referenced is a pointer
- the 2nd `const` -> that pointer itself cannot be changed through `param`
- the 1st `const int` -> the being-pointed-to int cannot be changed

# Function

### Return Type

##### `T&` vs `T`

| Aspect      | Choice | Why                                |
|:---|:---:|:---|
| `T&` vs `T`     | `T&`    | If you want chained-same operations.<br> For example: `std::cout << 'abc' << 123;`  |
|             | `T`     | Return a new value / independent    |

##### `std::optional<T>` vs `std::variant<T1, ...>`

| Aspect | `std::optional<T>` | `std::variant<Ts...>` |
| :---: | :---: | :---: |
| Meaning | Maybe one `T`, or no value (`nullopt`) | Exactly one of several types |
| Access | `has_value`, `value`, `*`, `value_or` | `std::visit`, `std::get`, `holds_alternative`, `index()` |
| Use when | Presence of one type is optional | You need one-of-many types |

### Handling `std::optional<T>`

1. (C++23) we can use monadic calls which is like chained function call in Python.

- For fallback, use `.value_or("... not found")`
```cpp
    std::string out = course.value_or("Course not found.");
```


- If you want to transform `std::optional<T>` -> `std::optional<U>`, 

    applies `.transform(lambda func)` before `.value_or()`

```cpp
    std::string out = course.transform(
        // map std:optional<Course> -> std::optimal<string>
        [](const Course& c) { return "Found: " + c.title; }
    )
```


- Remember to compile by: `g++ -std=c++23 main.cpp -o main`

2. (older C++) just use if-else:

```cpp
    auto course = db.find_course(title);

    if (course) {   // or: if (course.has_value())
        std::cout << course->title << "\n";
    } else {
        std::cout << "Course not found.\n";
    }
```

### `inline` func_name()

| Aspect | Inline Functions | Macros |
|:---:|:---:|:---:|
| Definition | Inline functions are functions defined with the `inline` keyword. | Macros are preprocessor directives defined using `#define`. |
| Scope | Inline functions have scope and type checking, like regular functions. | Macros have no scope or type checking. They are replaced by the preprocessor. |
| Evaluation of Arguments | Arguments are evaluated once. | Arguments may be evaluated multiple times (e.g., in expressions). |
| Handling | Inline functions are handled by the compiler. | Macros are handled by the preprocessor. |
| Private Members | Can access private members of a class. | Cannot access private members of a class. |
| Execution Overhead | Compiler may ignore the inline request if the function is too large. | Macros are always substituted into code. |
| Recursion | Inline functions can call themselves recursively. | Macros cannot be recursive. |


1. `inline` function is a *run-to-compile time optimization*, where code is expanded at the point of call at **compile time**. It's mostly used for function that does simple thing like outputing, summation, etc.


2. Inlining is a request, not guaranteed by the compiler.The compiler may ignore inlining if the function contains:
    - loops, recursion, static variables, switch/goto, or 
    - a non-void function without a return statement.

3. **`virtual` member function CANNOT do `inline`**

![inline-func](../_images/inline_function.png)

```cpp
inline int getSum(int a, int b){
    return a + b;
}

int main(){
    int num1 = 5, num2 = 10;

    int result = getSum(num1, num2);
    cout << "Sum: " << result << endl;
    return 0;
}
```

# Macros
`#define` vs `using`

Fundamentally, `#define` vs `using` is just alias of code (constant, function, expression) vs alias of variable name
- macro  = text substitution before compilation
- using = scoped aliasing / name import at compile time

```cpp
#define PI 3.14159
#define SQUARE(x) ((x) * (x))
#define LOG(msg) std::cout << msg << '\n'
```

```cpp
using ll = long long;` or `using std::cout;
```

# lvalue and rvalue (... references)

##### Normal: lvalue vs rvalue variables

- lvalue: has identity adress and name -> can accessing address `&x`
    - Can be placed **LHS or RHS** of any expression.
- rvalue: has NO identity, address, and name -> temporary value -> CANNOT use `&x`
    - Can **only be placed in RHS** of expression

```cpp
int val = 2;                    // 2 = rvalue;          val = lvalue
int* ptr = 0x42;                // 0x42 = rvalue;       ptr = lvalue
vector<int> v1{1, 2, 3};        // 1, 2, 3 = rvalue;    v1 = lvalue

auto v4 = v1 + v2;              // v1 + v2 = rvalue;    v1 = lvalue, v2 = lvalue, v4 = lvalue
v1 += v4                        //                      v4 = lvalue, v1 = lvalue (+= operator does not have v1 + v4)

size_t size = v.size();         // v.size() = rvalue;   size = lvalue
val = (int) size;               // (int) size = rvalue; size = lvalue, val = lvalue

a = b;                          //                      b = lvalue (copy assignment), a = lvalue
a = std::move(b);               // move(b) = rvalue;    b = lvalue, a = lvalue

ptr = &val                      // &val = rvalue;       val = lvalue, ptr = lvalue
v1[2] = *ptr;                   //                      *ptr = lvalue, ptr = lvalue, v1[2] = lvalue, v1 = lvalue
```

##### Harder: lvalue and rvalue references

1.  rvalue reference is super tricky:
    - rvalue reference **also has no identity / its own address**,

    - BUT, **rvalue reference is an lvalue**, meaning **you can still do &ref**. Why:
    
    ```cpp
    auto&& ref = 10;
    ```    
    When you write the above code:
    1. Compiler creates a "temporary object" in memory to hold that `10`.
    2. That "hidden" memory slot where `10` lives now has an address.
    3. Because you gave that memory slot a name (`ref`), it's now an lvalue.
    4. You can now do `&ref`, but it returns **the hidden address of `10`**

2. lvalue reference, despite the lvalue name, HAS NO IDENTITY / ITS OWN address.
    ```cpp
    int y = 10;
    int& x = y; 

    // There is no way in the C++ language to find the memory 
    // address of the 'link' x itself. 
    // You only ever see the address of y.
    ```

3. Binding: 
- rvalue reference can only point to rvalue, *extending its temporary lifetime*.

- **non-const** lvalue reference can only point to lvalue, *something having an identity / address*.

- CAN **bind `const` lvalue reference to rvalue** due to the weird BUT above:
    - By forcing `const`, C++ says <br>"You can look but can't change this temporary, so it doesn't matter that it's about to disappear or extended in lifetime"

    ```cpp
    auto& ptr2 = ptr;       // ptr2 = lvalue reference
    auto& ptr3 = &val;      // ERROR: can't bind lvalue ref. (ptr3) to rvalue (&val)

    auto&& v4 = v1 + v2;    // v4 = rvalue reference
    auto&& val2 = val;      // ERROR: can't bind rvalue ref. (val2) to lvalue (val)

    const auto& x = a + 5   // OK: can bind const, lvalue ref. (x) to rvalue (a + 4)
    ```

    ```cpp
    nonConst_l_ref(T& v);
    const_l_ref(const T& v);
    nonConst_r_ref(T&& v);
    // NO ONE USE const rvalue reference

    vector<int> v1 = v2 + v3;
    nonConst_l_ref(v1);         // OK
    const_l_ref(v1);            // OK
    nonConst_r_ref(v1);         // ERROR

    nonConst_l_ref(v2 + v3);    // ERROR
    const_l_ref(v2 + v3);       // OK
    nonConst_r_ref(v2 + v3);    // OK

    nonConst_l_ref(2);          // ERROR
    const_l_ref(2);             // OK
    nonConst_r_ref(2);    // OK    
    ```

- NO ONE USE const rvalue reference

# Class
- `struct`: public by default

- `class`: private by default

### `private`, `public`, `protected`

##### When defining in Base Class

- `private`: only for **this** class
- `protected`: only for **this** or **derived (child)** class
- `public`: can be accessed by **anyone**

##### When inheritance: `public` vs `private` vs protected`

1. Inheritance overrides Base's original access status

| Inheritance used by Child | Base `public` becomes in Child | Base `protected` becomes in Child | Base `private` becomes in Child |
|:---:|:---:|:---:|:---:|
| Child : public Base| public | protected | not accessible |
| Child : protected Base | protected | protected | not accessible |
| Child : private Base | private | private | not accessible |

2. Given the final status after inherited:
- If it's public: Child, Grandchild, Outsider aka anyone can access
- If it's protected: only Child and Grandchild / Descendants
- If it's private: only (first-order) Child

    ```cpp
    struct Base {
        public: void pub() {...}
    };
    struct Child_Public : public Base {};
    struct Child_Private : private Base {};

    int main() {
        Base b;
        b.pub();            // always OK

        Child_Public cp;
        cp.pub();           // OK

        Child_Private cpr;
        cpr.pub();          // ERROR
    }
    ```

### Child has same field / member variable name as Base

1. There's no such thing as `virtual` member variable / field.

2. If a Child has member variable / field with *same name* as Base's,<br>**it hides Base::member_var**:

    ```cpp
    struct A { int a; };
    struct B : public A {
        double a; // hides A::a 
    }

    B child;
    std::cout << child.a;       // B::a = 3.14
    std::cout << child.A::a;    // A::a = 42
    ```

### `this`

`this` is a **pointer** (storing address) of the current instance of a class
```cpp
    class X {
        int value;
        public:
            void set(int v) {
                this->value = v;
            }
    };
```

### `explicit` and automatic type conversion

`explicit` helps avoid automatic type conversion,

```cpp
    explicit ClassName(T* p) : ptr{p} {}
```

because: any non-`explicit` class that has a constructor taking an argument of another class will have automatic conversion when the equals sign appears, left-hand side having the class name, and the right-hand side having the argument type.

For example, if:
```cpp
    ClassName(int p) : ptr{p} {}
    ClassName(int* p) : ptr{p} {}
```

Then these will do automatic type conversation:
```cpp
    ClassName x = 5;            // equivalent to `ClassName(5);`
    ClassName x = new int{5};   // equivalent to 'ClassName(new int{5});
```

### `virtual`

##### `virtual` member function

`virtual` function CANNOT do `inline`

```cpp
class BaseClass {
public:
    /* Pure virtual function: any derived ChildClass.
       Must has its own implementation */
   void foo() = 0; // or `virtual void foo() = 0` i.e `= 0` means virtual

    /* Provide a default implementation in the base class. 
       Derived child have the option to override it or not */
    virtual void foo2() {
        // your work here
    }

    /* `virtual` function exists = must have `virtual` destructor
        so it's safe when deleting an instance of a derived class */ 
    virtual ~BaseClass() {}

    /* Virtual for polymorphism (inheritance / override), but slower runtime.
       Non-Virtual for behavior that should never change, but faster runtime */
    void foo3() { ... };
};
```

##### `virtual` inheritance

1. **A derived class does not have to inherit virtually just to override a function.**
    Virtual inheritance is only needed when you want one shared A subobject in a diamond, like A -> B and A -> C and then D -> B, C. 
    
    It is not a requirement for overriding virtual functions.

    We want:

    ```py
    """
     A
    / \
    B C
    \ /
     D
    """
    ```

    not
    ```py
    """
    A A
    | |
    B C
    \ /
     D 
    """
    ```

2. If both B and C override the same virtual function from A, D may need to resolve the ambiguity.
That means D often provides its own override and explicitly chooses B’s or C’s implementation. 

    ```cpp
    struct A {
        virtual void f();
    };

    struct B : virtual A {
        void f() override;
    };

    struct C : virtual A {
        void f() override;
    };

    struct D : B, C {
        void f() override {
            B::f();   // or C::f()
        }
    };    
    ```

    Virtual inheritance does not automatically pick one. **Anything otherwise results in compile error**

`= default`

### `= 0`, `= default`, `= delete`, or nothing

| Form | Meaning | is Virtual? |
|:---:|:---:|:---:|
| `= 0` | Pure virtual function, makes the class abstract | Yes, auto by definition |
| `= default` | Ask compiler to generate the normal implementation | Only if you add `virtual`|
| `= delete` | Forbid from being written in derived class or being called | No, `virtual` is uesless|
| nothing | Normal function; may or may not be overridden only if it is virtual. | No |

### Copy (constructor and assignment)

1. Copy is for new initialize variable copying existing one. E.g: `int b = a`.

2. Assignment is for **already init** variable copying existing one. E.g: `b = a`.

3. Both results in the same thing (`b` and `a` separated memory but same values), but <br>
**assignment allows chained-same operation**

##### Copy Constructor
- Copy each **non-pointer, non-container** member field of other class

- For **pointer**, **container**, perform deepcopy

```cpp
    X a;
    X b = a;   // copy constructor
    X c(a);    // copy constructor
```

##### Copy Assignment (Operator)

```cpp
X& operator=(const X& other) {
    if (this != &other) {
        // Do something
    }
    return *this;
}
```

Return `X&` since don't waste another copy of returned value, and <br> allow **chained operation for already init variable**

```cpp
    X a;
    X b;
    X c;
    b = a = c;     // copy assignment: (b = a) = c
```

##### Implementation (rule of 3) unifying approach pointer, container, and value
If `copy` implemented, we want all.
- Deferencing

- Copy Constructor

- Copy Assignment Operator

```cpp
#include <algorithm> // for std::copy
#include <vector>

class X {
    int* nums = nullptr;
    size_t size = 0;
    std::vector<int> vec; // Imagine you also have a container

public:
    X() = default;

    ~X() { delete[] nums; }

    // 2. UNIFIED COPY CONSTRUCTOR
    X(const X& other) : 
        size(other.size), 
        nums(nullptr),
        vec(other.vec)  // 1. Containers copy automatically via initializer list
    {
        // 2. Raw Pointers require manual deep copy logic
        if (other.nums) {
            nums = new int[size];
            std::copy(other.nums, other.nums + size, nums);
        }
    }

    // 3. UNIFIED COPY ASSIGNMENT
    X& operator=(const X& other) {
        if (this != &other) {
            // A. Handle Raw Pointers (Manual Deep Copy first for safety)
            int* new_nums = nullptr;
            if (other.nums) {
                new_nums = new int[other.size];
                std::copy(other.nums, other.nums + other.size, new_nums);
            }

            // B. Handle Containers (They handle their own safety/assignment)
            vec = other.vec; 

            // C. Commit changes to pointers
            delete[] nums;
            nums = new_nums;
            size = other.size;
        }
        return *this; // Don't forget to return *this!
    }
};
```

### Move (constructor and assignment)

- How to invoke:
    ```cpp
    X a = X();           // MOVE CONSTRUCTOR (X() is a temporary rvalue)
    X b = std::move(a);  // MOVE CONSTRUCTOR (std::move forces a to be an rvalue)

    X a;
    a = X();             // MOVE ASSIGNMENT (X() is a temporary rvalue)
    X b;
    a = std::move(b);    // MOVE ASSIGNMENT (std::move forces b to be an rvalue)
    ```

- vs Copy:

    1. Same: rationale for why assignment exists vs constructor.

    2. Diff:
        - Move arg must be **rvalue reference** (not lvalue refenrece)

        - Instead of copying, it steals everything (pointer and values) from input arg (making it null)

##### Implementation (Rule of 5) for both pointer and containers

Rule of 5: if touching `move`, implemented all
- Move (constructor, assignment)
- Copy (constructor, assignment)
- Destructor

```cpp
#include <utility>   // for std::move and std::exchange
#include <vector>

class X {
    int* nums = nullptr;      // Pointer (needs manual nulling)
    std::vector<int> vec;     // Container (has its own move logic)

public:
    /* Same as rule of 3 */
    X() = default;
    ~X();
    X(const X& other);
    X& operator=(const X& other);    

    // UNIFIED MOVE CONSTRUCTOR
    X(X&& other) noexcept 
        : nums(std::exchange(other.nums, nullptr)), // Steal & null pointer
          vec(std::move(other.vec))                 // Move the vector
    {}

    // UNIFIED MOVE ASSIGNMENT
    X& operator=(X&& other) noexcept {
        if (this != &other) {
            delete[] nums; // Clean up old pointer memory

            // The Pattern:
            nums = std::exchange(other.nums, nullptr); // Pointer pattern
            vec  = std::move(other.vec);               // Container pattern
        }
        return *this;
    }
};
```

### Working with header `.h` files

In `.cpp` file, remember to use `ClassName::` when writing implementation of member function

# Namespace

Namespace can be user-defined without `using`

```cpp
namespace mylib {
    int add(int a, int b) { return a + b; }
}

int f() {
    return mylib::add(2, 3); // OK w/o `using namespace std`
}
```

### When to use: Class vs Namespace vs Enum

"state management" or "just to avoid name collison across modules / libraries", "without restrition of Enum"

1. Use a namespace when:
- You want to group related utilities/APIs.
- You want to prevent name conflicts across modules/libraries.
- You do not need object state.

    Example:
    ```cpp
    namespace math_utils {
        double pi = 3.1415926535;
        double square(double x) { return x * x; }
    }
    ```

2. Use a class when:
- You need a custom type with state.
- You want encapsulation and invariants.
- You want OOP features (inheritance/interfaces/polymorphism).

    Example:
    ```cpp
    class BankAccount {
    private:
        double balance;

    public:
        explicit BankAccount(double b) : balance(b) {}
        void deposit(double x) { balance += x; }
        double getBalance() const { return balance; }
    };
    ```


Tips: Put classes inside a namespace to keep your project organized.
```cpp
namespace finance {
    class BankAccount { /* ... */ };
}
```




Rule of thumb:
- If your question is “how do I group code names?” -> namespace.
- If your question is “how do I model an entity/object?” -> class.

```cpp

```

# Template

### `<typename T>` vs `<typename T, typename D>`

1. 

- What changes when you add D: derived variable must provides D type (unless D has a default).
    ```cpp
    template<typename T, typename D>
    struct X { T value1; D value2; };

    X<int, double> a;   // OK
    X<int> a;           // NOT OK
    ```

- If you want D optional, give a default:

    ```cpp
    template<typename T, typename D = void>
    struct X { T value1; D value; };

    X<int, double> a;   // OK
    X<int> a;           // OK
    ```

2. You can declare template w/ 2 params, and only use 1 in definition.

    ```cpp
    template<typename T, typename D>
    struct X {
        T value;   // D is not used here
    };
    ```

### Concept: restrict / require what `typename` can only be

```cpp
template<class T1, ...>
concept ConceptName = bool_FuncName<T1, ...>
```

or

```cpp
template<class T1, ...>
concept ConceptName = bool_FuncName<T1, ...> && / || ...
```

**How to use:**  these are equivalent:

```cpp
template <ConceptName T>
void f(T ...) { ... }
```

```cpp
template<class T>
requires ConceptName<T>
void f(T x);
```

```cpp
template<class T>
void f(T x) requires ConceptName<T>;
```

Yes, if they express the same constraint on the same template parameters, they are semantically equivalent.

For example, these are equivalent:



In [ ]:
template<class T>
void f(T x) requires ConceptName<T>;







All three mean: only participate in overload resolution when `ConceptName<T>` is satisfied.

Only differences are style and where the constraint is written.

- Templates are “generic code.”
- Concepts are “requirements for what types are allowed into that generic code.”

-> **Concept helps detecting error early in compile error before runtime logic (costly)**

1. Without concepts, template errors can appear unintentionally when operators not available in that data type.

    ```cpp
    template<typename T>
    T add(T a, T b) {
        return a + b;  // error if T doesn't support +
    }
    ```

2. With a concept: now only “addable” types are accepted.

    ```cpp
    #include <concepts>

    template<typename T>
    concept Addable = requires(T x, T y) { x + y; };

    template<Addable T>
    T add(T a, T b) {
        return a + b;
    }
    ```

3. Equivalent shorthand (for example):

    ```cpp
    template<std::integral T>
    T gcd(T a, T b) { /* ... */ }
    ```

### Template vs Derived Class

It's just **static** vs **dynamic** polymorphism

| Prefer templates when | Prefer derived classes when |
|:---:|:---:|
| **Runtime** efficiency is most important | **Compile-time** efficiency is most important |
| No common base defined | Want to hide implementations |
| | Don’t want code bloat |

# lvalue and rvalue (... references)

##### Normal: lvalue vs rvalue variables

- lvalue: has identity adress and name -> can accessing address `&x`
    - Can be placed **LHS or RHS** of any expression.
- rvalue: has NO identity, address, and name -> temporary value -> CANNOT use `&x`
    - Can **only be placed in RHS** of expression

```cpp
int val = 2;                    // 2 = rvalue;          val = lvalue
int* ptr = 0x42;                // 0x42 = rvalue;       ptr = lvalue
vector<int> v1{1, 2, 3};        // 1, 2, 3 = rvalue;    v1 = lvalue

auto v4 = v1 + v2;              // v1 + v2 = rvalue;    v1 = lvalue, v2 = lvalue, v4 = lvalue
v1 += v4                        //                      v4 = lvalue, v1 = lvalue (+= operator does not have v1 + v4)

size_t size = v.size();         // v.size() = rvalue;   size = lvalue
val = (int) size;               // (int) size = rvalue; size = lvalue, val = lvalue

a = b;                          //                      b = lvalue (copy assignment), a = lvalue
a = std::move(b);               // move(b) = rvalue;    b = lvalue, a = lvalue

ptr = &val                      // &val = rvalue;       val = lvalue, ptr = lvalue
v1[2] = *ptr;                   //                      *ptr = lvalue, ptr = lvalue, v1[2] = lvalue, v1 = lvalue
```

##### Harder: lvalue and rvalue references

1.  rvalue reference is super tricky:
    - rvalue reference **also has no identity / its own address**,

    - BUT, **rvalue reference is an lvalue**, meaning **you can still do &ref**. Why:
    
    ```cpp
    auto&& ref = 10;
    ```    
    When you write the above code:
    1. Compiler creates a "temporary object" in memory to hold that `10`.
    2. That "hidden" memory slot where `10` lives now has an address.
    3. Because you gave that memory slot a name (`ref`), it's now an lvalue.
    4. You can now do `&ref`, but it returns **the hidden address of `10`**

2. lvalue reference, despite the lvalue name, HAS NO IDENTITY / ITS OWN address.
    ```cpp
    int y = 10;
    int& x = y; 

    // There is no way in the C++ language to find the memory 
    // address of the 'link' x itself. 
    // You only ever see the address of y.
    ```

3. Binding: 
- rvalue reference can only point to rvalue, *extending its temporary lifetime*.

- **non-const** lvalue reference can only point to lvalue, *something having an identity / address*.

- CAN **bind `const` lvalue reference to rvalue** due to the weird BUT above:
    - By forcing `const`, C++ says <br>"You can look but can't change this temporary, so it doesn't matter that it's about to disappear or extended in lifetime"

    ```cpp
    auto& ptr2 = ptr;       // ptr2 = lvalue reference
    auto& ptr3 = &val;      // ERROR: can't bind lvalue ref. (ptr3) to rvalue (&val)

    auto&& v4 = v1 + v2;    // v4 = rvalue reference
    auto&& val2 = val;      // ERROR: can't bind rvalue ref. (val2) to lvalue (val)

    const auto& x = a + 5   // OK: can bind const, lvalue ref. (x) to rvalue (a + 4)
    ```

    ```cpp
    nonConst_l_ref(T& v);
    const_l_ref(const T& v);
    nonConst_r_ref(T&& v);
    // NO ONE USE const rvalue reference

    vector<int> v1 = v2 + v3;
    nonConst_l_ref(v1);         // OK
    const_l_ref(v1);            // OK
    nonConst_r_ref(v1);         // ERROR

    nonConst_l_ref(v2 + v3);    // ERROR
    const_l_ref(v2 + v3);       // OK
    nonConst_r_ref(v2 + v3);    // OK

    nonConst_l_ref(2);          // ERROR
    const_l_ref(2);             // OK
    nonConst_r_ref(2);    // OK    
    ```

- NO ONE USE const rvalue reference

# -

# Algo

### Swap

1. Swap big containers (e.g `std::string`, `std::vector`)

```cpp
    void swap(T& a, T& b) {
        T temp = std::move(a);
        a = std::move(b);
        b = std::move(temp);
    }
```

2. Swap small stuff (e.g pointer, `int`), `std::move()` is not boosting

```cpp
    void swap(T& a, T& b) {
        T temp = a;
        a = b;
        b = temp;
    }
```

# -

# Data Structures

### Vector

##### `.emplace_back()` vs `.push_back()`

It matters with custom Class

```cpp
class CustomClass {
    std::string str; int val;
};

std::vector<CustomClass> v;
v.push_back(CustomClass("a", 1));
v.emplace_back("a", 1);
```